## Curricular Unit: Information Integration and Analytic Data Processing
## Professor: Márcia Barros

### Group 5

### Nuno Lages 66091
### Nuno Rosado 66104
### Diogo Carvalho 66123

---

## Imports

In [1]:
import json
import os
import math
import random
import pickle
import unicodedata
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import GradientBoostingRegressor
from rank_bm25 import BM25Okapi
from groq import Groq

---
# Phase 1: Data Warehouse

## Step 3: ETL Process

### Extraction

In [2]:
# ===============================
# CONFIGURATION
# ===============================

RUN_ENV = "local"     # options: "colab" or "local"

# Colab configuration
COLAB_BASE_DIR = "/content/drive/My Drive/FCUL-MEI/Semestre_2/IPAI/Projeto"

# Local configuration
LOCAL_BASE_DIR =   "./"  # change to your local folder if needed

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

    base_path = Path(COLAB_BASE_DIR)

elif RUN_ENV == "local":
    base_path = Path(LOCAL_BASE_DIR)

else:
    raise ValueError("RUN_ENV must be 'colab' or 'local'")

# Create base directory if needed
base_path.mkdir(parents=True, exist_ok=True)
# Create data directory if needed
(base_path / "data").mkdir(parents=True, exist_ok=True)

# File paths
SONGS = base_path / "data" / "merged_songs.csv"
ARTISTS = base_path / "data" / "artists.csv"
REVIEWS = base_path / "data" / "pitchfork_reviews.csv"

# Dimension paths
dim_music_file = base_path / "data" / "dim_music.tsv"
dim_date_file = base_path / "data" / "dim_date.tsv"
dim_genre_file = base_path / "data" /"dim_genre.tsv"
dim_album_file = base_path / "data" /"dim_album.tsv"
dim_review_file = base_path / "data" / "dim_review.tsv"
dim_artist_file = base_path / "data" / "dim_artist.tsv"

# Fact table path
fact_music_file = base_path / "data" / "fact_music.tsv"

# Lookup files
music_lookup_file = base_path / "data" / "lk_music.json"
date_lookup_file = base_path / "data" / "lk_date.json"
genre_lookup_file = base_path / "data" / "lk_genre.json"
album_lookup_file = base_path / "data" / "lk_album.json"
review_lookup_file = base_path / "data" / "lk_review.json"
artist_lookup_file = base_path / "data" / "lk_artist.json"

### Transformation

In [3]:
songs_df = pd.read_csv(SONGS, sep=',')
artists_df = pd.read_csv(ARTISTS, sep=',')
reviews_df = pd.read_csv(REVIEWS, sep=';')

# Songs dataset cleaning
songs_df.info()
songs_df = songs_df.dropna(subset=['album_name'])
songs_df = songs_df.dropna(subset=['release_date'])
songs_df = songs_df.drop_duplicates(subset=['id'])
# Maintain only the entries that have a complete date
songs_df = songs_df[songs_df['release_date'].str.len() == 10]
songs_df = songs_df[~(songs_df['lyrics'].str.contains(r'\[Instrumental\]', na=False))]

# Artists dataset cleaning
artists_df.info()
artists_df = artists_df.dropna()
artists_df = artists_df.drop_duplicates(subset=['name'])

# Reviews dataset cleaning
reviews_df.info()
reviews_df = reviews_df.dropna()
reviews_df = reviews_df.drop_duplicates()

# Convert the list of genres into a real list
artists_df['genres'] = artists_df['genres'].apply(ast.literal_eval)

# Text normalization
songs_df['album_name'] = songs_df['album_name'].str.lower().str.strip()
reviews_df['album'] = reviews_df['album'].str.lower().str.strip()

songs_df['artists'] = songs_df['artists'].str.lower().str.strip()
reviews_df['artist'] = reviews_df['artist'].str.lower().str.strip()
artists_df['name'] = artists_df['name'].str.lower().str.strip()

songs_df['genre'] = songs_df['genre'].str.lower().str.strip()
reviews_df['genre'] = reviews_df['genre'].str.lower().str.strip()
artists_df['main_genre'] = artists_df['main_genre'].str.lower().str.strip()
artists_df['genres'] = artists_df['genres'].apply(
    lambda x: [g.lower().strip() for g in x]
)

# Convert Review date to datetime (DD/MM/AAAA)
reviews_df['review_date'] = pd.to_datetime(
    reviews_df['review_date'],
    errors='coerce'
)
reviews_df['review_date'] = reviews_df['review_date'].dt.strftime('%d/%m/%Y')


# Kepp only reviews of the existing albums (9012 Reviews)
albums = songs_df['album_name'].dropna().unique()
reviews_df = reviews_df[reviews_df['album'].isin(albums)]


# ===============================
# EXPORT CLEANED FILES
# ===============================

songs_df.to_csv(base_path / "data" / "clean_songs.csv", index=False)
artists_df.to_csv(base_path / "data" / "clean_artists.csv", index=False)
reviews_df.to_csv(base_path / "data" / "clean_reviews.csv", index=False)

print("\nCleaned files saved successfully.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47472 entries, 0 to 47471
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    47472 non-null  object 
 1   release_date            41217 non-null  object 
 2   id                      47472 non-null  object 
 3   album_name              47472 non-null  object 
 4   artists                 47472 non-null  object 
 5   danceability            47472 non-null  float64
 6   energy                  47472 non-null  float64
 7   key                     47472 non-null  int64  
 8   loudness                47472 non-null  float64
 9   mode                    47472 non-null  int64  
 10  speechiness             47472 non-null  float64
 11  acousticness            47472 non-null  float64
 12  instrumentalness        47472 non-null  float64
 13  liveness                47472 non-null  float64
 14  valence                 47472 non-null

# Loading

In [4]:
# ---------- Helpers ----------

def load_lookup_table(dimension_name: str):
    """Load the natural key -> surrogate key mapping for a dimension."""
    file_name = f"lut_{dimension_name}.json"
    if os.path.isfile(file_name):
        with open(file_name, "r", encoding="utf-8") as f:
            lookup = json.load(f)
        # JSON values may already be integers, but enforce it just in case.
        lookup = {k: int(v) for k, v in lookup.items()}
    else:
        lookup = {}

    next_surrogate_key = (max(lookup.values()) + 1) if lookup else 1
    return lookup, next_surrogate_key


def save_lookup_table(dimension_name: str, lookup_table: dict):
    """Persist the lookup table to disk."""
    file_name = f"lut_{dimension_name}.json"
    with open(file_name, "w", encoding="utf-8") as f:
        json.dump(lookup_table, f, ensure_ascii=False, indent=2)


def save_dimension(df: pd.DataFrame, file_name: str):
    """Append new rows to a dimension TSV file, creating the header if needed."""
    write_header = not os.path.isfile(file_name)
    mode = "w" if write_header else "a"

    if not df.empty:
        df.to_csv(file_name, sep="\t", index=False, mode=mode, header=write_header)


def save_fact(df: pd.DataFrame, file_name: str):
    """Append fact rows to the fact TSV file, creating the header if needed."""
    write_header = not os.path.isfile(file_name)
    mode = "w" if write_header else "a"
    df.to_csv(file_name, sep="\t", index=False, mode=mode, header=write_header)


def assign_surrogate_keys(natural_keys: pd.Series, lookup_table: dict, next_surrogate_key: int):
    """Assign surrogate keys to a series of natural keys.

    Returns:
        key_series: surrogate key for each row in natural_keys
        new_rows_df: dataframe with only the newly discovered natural keys
        updated_lookup_table
        updated_next_surrogate_key
    """
    new_keys = [key for key in pd.unique(natural_keys) if key not in lookup_table]

    if new_keys:
        new_mapping = {key: sk for sk, key in enumerate(new_keys, start=next_surrogate_key)}
        lookup_table.update(new_mapping)
        next_surrogate_key += len(new_keys)
        new_rows_df = pd.DataFrame({
            "natural_key": new_keys,
            "surrogate_key": [lookup_table[key] for key in new_keys]
        })
    else:
        new_rows_df = pd.DataFrame(columns=["natural_key", "surrogate_key"])

    key_series = natural_keys.map(lookup_table).astype(int)
    return key_series, new_rows_df, lookup_table, next_surrogate_key

### Music Dimension

In [5]:
def build_music_dimension(df_songs: pd.DataFrame):
    lookup, next_sk = load_lookup_table("music")

    music_keys, new_songs, lookup, next_sk = assign_surrogate_keys(
        df_songs["id"], lookup, next_sk
    )

    dim_music = new_songs.rename(columns={
        "surrogate_key": "MUSIC_KEY",
        "natural_key": "ID"
    })

    if not dim_music.empty:
        music_attributes = (
            df_songs[["id", "name", "lyrics"]]
            .drop_duplicates(subset=["id"])
        )

        dim_music = (
            dim_music.merge(music_attributes, left_on="ID", right_on="id", how="left")
            [["MUSIC_KEY", "ID", "name", "lyrics"]]
        )

    save_dimension(dim_music, dim_music_file)
    save_lookup_table("music", lookup)

    return music_keys, dim_music, lookup

In [6]:
music_keys, dim_music, lookup_song = build_music_dimension(songs_df)
print("Dimensão de Música construída e guardada!")

Dimensão de Música construída e guardada!


### Date Dimension

In [7]:
def get_season(month):
    if month in [12, 1, 2]: return "Winter"
    if month in [3, 4, 5]: return "Spring"
    if month in [6, 7, 8]: return "Summer"
    return "Autumn"

def build_date_dimension(df_songs: pd.DataFrame, df_reviews: pd.DataFrame):
    lookup, next_sk = load_lookup_table("date")

    # 1. Extrair datas das reviews
    rev_dates = pd.to_datetime(df_reviews["review_date"], errors='coerce').dropna()

    # 2. Extrair datas das músicas
    song_dates = pd.to_datetime(df_songs["release_date"], errors='coerce').dropna()

    # 3. Concatenar ambas e obter apenas a parte da data (YYYY-MM-DD) única
    all_dates = pd.concat([rev_dates, song_dates]).dt.date.unique()

    unique_dates_str = [str(d) for d in all_dates]

    df_dates = pd.DataFrame({"date_raw": unique_dates_str})

    # 3. Atribuir Surrogate Keys
    date_keys, new_dates, lookup, next_sk = assign_surrogate_keys(
        df_dates["date_raw"], lookup, next_sk
    )

    # 4. Renomear colunas base
    dim_date = new_dates.rename(columns={
        "surrogate_key": "Date_ID",
        "natural_key": "Complete Date"
    })

    if not dim_date.empty:
        dt = pd.to_datetime(dim_date["Complete Date"])

        dim_date["Full Date (Extended Format)"] = dt.dt.strftime('%B %d, %Y')
        dim_date["Standard American Date"] = dt.dt.strftime('%m/%d/%Y')
        dim_date["Day of Month"] = dt.dt.day
        dim_date["Day of Year"] = dt.dt.dayofyear
        dim_date["Day of Week"] = dt.dt.day_name()
        dim_date["Week of Year"] = dt.dt.isocalendar().week.astype(int)
        dim_date["Month Description"] = dt.dt.month_name()
        dim_date["Month Abbreviation"] = dt.dt.strftime('%b')
        dim_date["Month Number"] = dt.dt.month
        dim_date["Quarter"] = dt.dt.quarter
        dim_date["Semester"] = dt.dt.month.apply(lambda x: 1 if x <= 6 else 2)
        dim_date["Year"] = dt.dt.year
        dim_date["Decade"] = (dt.dt.year // 10) * 10
        dim_date["Season"] = dt.dt.month.apply(get_season)

        ordered_cols = [
            "Date_ID", "Complete Date", "Full Date (Extended Format)",
            "Standard American Date", "Day of Month", "Day of Year",
            "Day of Week", "Week of Year", "Month Description",
            "Month Abbreviation", "Month Number", "Quarter",
            "Semester", "Year", "Decade", "Season"
        ]

        dim_date = dim_date[ordered_cols]

    save_dimension(dim_date, dim_date_file)
    save_lookup_table("date", lookup)

    return date_keys, dim_date, lookup

date_keys, dim_date, lookup_date = build_date_dimension(songs_df, reviews_df)

### Artist Dimension

In [8]:
def build_artist_dimension(df_artists: pd.DataFrame):
    lookup, next_sk = load_lookup_table("artist")

    artist_keys, new_artists, lookup, next_sk = assign_surrogate_keys(
        df_artists["name"], lookup, next_sk
    )

    dim_artist = new_artists.rename(columns={
        "surrogate_key": "Artist_ID",
        "natural_key": "Name"
    })

    if not dim_artist.empty:
        dim_artist = dim_artist.merge(
            df_artists[["name", "followers", "popularity"]],
            left_on="Name", right_on="name", how="left"
        ).drop(columns=["name"]).rename(columns={
            "followers": "Followers",
            "popularity": "Artist Popularity"
        })

        cols = ["Artist_ID", "Name", "Followers", "Artist Popularity"]
        dim_artist = dim_artist[cols]

    save_dimension(dim_artist, dim_artist_file)
    save_lookup_table("artist", lookup)

    return artist_keys, dim_artist, lookup

In [9]:
artist_keys, dim_artist, lookup_artist = build_artist_dimension(artists_df)

In [10]:
dim_artist = dim_artist.drop_duplicates(subset=['Artist_ID'])

### Genre Dimension

In [11]:
def build_genre_dimension(df_songs: pd.DataFrame):
    lookup, next_sk = load_lookup_table("genre")

    unique_genres = df_songs["genre"].dropna().unique()

    genre_keys, new_genres, lookup, next_sk = assign_surrogate_keys(
        pd.Series(unique_genres), lookup, next_sk
    )

    dim_genre = new_genres.rename(columns={
        "surrogate_key": "Genre_ID",
        "natural_key": "Genre name"
    })

    if not dim_genre.empty:
        dim_genre = dim_genre[["Genre_ID", "Genre name"]]

    save_dimension(dim_genre, dim_genre_file)
    save_lookup_table("genre", lookup)

    return genre_keys, dim_genre, lookup

In [12]:
genre_keys, dim_genre, lookup_genre = build_genre_dimension(songs_df)

### Album e Review Dimensions

In [13]:
def build_album_review_dimensions(df_songs: pd.DataFrame, df_reviews: pd.DataFrame):
    lk_alb, next_sk_alb = load_lookup_table("album")

    df_songs['alb_nat_key'] = df_songs['album_name'].astype(str) + "||" + df_songs['artists'].astype(str)

    album_keys, new_albs, lk_alb, next_sk_alb = assign_surrogate_keys(
        df_songs["alb_nat_key"], lk_alb, next_sk_alb
    )

    dim_album = pd.DataFrame({
        "ALB_NAT_KEY": list(lk_alb.keys()),
        "Album_ID": list(lk_alb.values())
    })

    album_names = df_songs[["alb_nat_key", "album_name"]].drop_duplicates("alb_nat_key")
    dim_album = dim_album.merge(album_names, left_on="ALB_NAT_KEY", right_on="alb_nat_key", how="left")

    dim_album = dim_album.rename(columns={"album_name": "Name"})[["Album_ID", "Name"]]

    dim_album.to_csv(dim_album_file, sep="\t", index=False)
    save_lookup_table("album", lk_alb)


    lk_rev, next_sk_rev = load_lookup_table("review")
    df_reviews['rev_nat_key'] = df_reviews['album'].astype(str) + "||" + df_reviews['artist'].astype(str)

    review_keys, new_revs, lk_rev, next_sk_rev = assign_surrogate_keys(
        df_reviews['rev_nat_key'], lk_rev, next_sk_rev
    )

    dim_review = new_revs.rename(columns={"surrogate_key": "Review_ID", "natural_key": "REV_NAT_KEY"})

    if not dim_review.empty:
        df_reviews['Album_ID'] = df_reviews['rev_nat_key'].map(lk_alb)

        dim_review = dim_review.merge(
            df_reviews[["rev_nat_key", "Album_ID", "reviewer", "label", "summary", "review", "score"]],
            left_on="REV_NAT_KEY", right_on="rev_nat_key", how="left"
        ).rename(columns={
            "score": "Classification",
            "reviewer": "Reviewer",
            "label": "Label",
            "summary": "Summary",
            "review": "Review"
        })

    cols_to_drop = ["REV_NAT_KEY", "rev_nat_key"]
    dim_review = dim_review.drop(columns=[c for c in cols_to_drop if c in dim_review.columns], errors='ignore')
    
    # Garante que não há colunas duplicadas por erro de merge (acontece se houver nomes iguais)
    dim_review = dim_review.loc[:, ~dim_review.columns.duplicated()]

    save_dimension(dim_review, dim_review_file)
    save_lookup_table("review", lk_rev)

    return lk_alb, lk_rev, dim_album, dim_review

In [14]:
lk_album, lk_review, dim_album, dim_review = build_album_review_dimensions(songs_df, reviews_df)

In [15]:
dim_review = dim_review.drop_duplicates(subset=['Review_ID'])

### Fact Table

In [16]:
def build_music_fact(df_songs, df_reviews, lks):
    # 1. Criar uma cópia para não alterar o DataFrame original
    fact = df_songs.copy()

    # 2. Mapeamento das Chaves Estrangeiras (FKs)
    # Music_ID e Artist_ID
    fact["Music_ID"] = fact["id"].map(lks['music'])
    fact["Artist_ID"] = fact["artists"].map(lks['artist'])

    # Genre_ID
    fact["Genre_ID"] = fact["genre"].map(lks['genre'])

    # Album_ID (usando a mesma lógica de chave composta da dimensão album)
    fact["album_nat_key"] = fact["album_name"].astype(str) + "||" + fact["artists"].astype(str)
    fact["Album_ID"] = fact["album_nat_key"].map(lks['album'])

    # 3. Mapeamento da Date_ID (Data da Música)
    # Garantir que a data está em formato string 'YYYY-MM-DD' para o lookup
    fact["release_date_str"] = pd.to_datetime(fact["release_date"], errors='coerce').dt.date.astype(str)
    fact["Date_ID"] = fact["release_date_str"].map(lks['date'])

    # 4. Tratamento de Métricas e Renomeação
    cols_mapping = {
        "duration_ms": "Duration",
        "danceability": "Danceability",
        "energy": "Energy",
        "loudness": "Loudness",
        "speechiness": "Speechiness",
        "acousticness": "Acousticness",
        "instrumentalness": "Instrumentalness",
        "liveness": "Liveness",
        "popularity": "Music popularity"
    }
    fact = fact.rename(columns=cols_mapping)

    # 5. Cálculo de Word Count (Baseado na coluna lyrics)
    fact["Word count"] = fact["lyrics"].fillna("").apply(lambda x: len(str(x).split()))

    # 6. Seleção final de colunas e limpeza
    final_cols = ["Music_ID", "Artist_ID", "Album_ID", "Date_ID", "Genre_ID"] + \
                 list(cols_mapping.values()) + ["Word count"]

    # Remover linhas onde não foi possível mapear a Música (integridade referencial)
    fact_music = fact[final_cols].dropna(subset=["Music_ID"])

    # Converter IDs para inteiros para evitar o formato float (ex: 1.0)
    id_cols = ["Music_ID", "Artist_ID", "Album_ID", "Date_ID", "Genre_ID"]
    fact_music[id_cols] = fact_music[id_cols].fillna(0).astype(int)

    # 7. Gravar o ficheiro
    save_fact(fact_music, fact_music_file)
    return fact_music

# --- EXECUÇÃO ---

# Certifique-se de que os lookups estão atualizados
lks = {
    'music': lookup_song,
    'artist': lookup_artist,
    'genre': lookup_genre,
    'album': lk_album,
    'date': lookup_date
}

fact_music = build_music_fact(songs_df, reviews_df, lks)
print("Tabela de Factos construída com sucesso!")

Tabela de Factos construída com sucesso!


## Bonus: Physical Design of the Warehouse

In [17]:
import sqlalchemy
from sqlalchemy import create_engine

# Configurações de acesso
USER = 'postgres'
PASSWORD = 'mestrado'
HOST = '127.0.0.1'
PORT = '5432'
DB_NAME = 'postgres'

# Criar o motor de conexão
engine = create_engine(f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}')

In [18]:
def upload_to_postgress():
  try:
    with engine.connect() as conn:
        print("Ligação OK")
  except Exception as e:
    print(e)

upload_to_postgress()

Ligação OK


In [19]:
def sanitize_columns(df):
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    return df_clean

def upload_to_postgres_robust():
    tables = {
        'dim_music': dim_music,
        'dim_artist': dim_artist,
        'dim_date': dim_date,
        'dim_genre': dim_genre,
        'dim_album': dim_album,
        'dim_review': dim_review,
        'fact_music': fact_music
    }
    
    print("A iniciar upload para o PostgreSQL")
    for name, df in tables.items():
        try:
            df_sql = sanitize_columns(df)
            df_sql.to_sql(name, engine, if_exists='replace', index=False)
            print(f"{name}: {len(df_sql)} linhas carregadas.")
        except Exception as e:
            print(f"Erro em {name}: {e}")

upload_to_postgres_robust()

A iniciar upload para o PostgreSQL
dim_music: 0 linhas carregadas.
dim_artist: 0 linhas carregadas.
dim_date: 0 linhas carregadas.
dim_genre: 0 linhas carregadas.
dim_album: 6381 linhas carregadas.
dim_review: 0 linhas carregadas.
fact_music: 40593 linhas carregadas.


### Indexs

In [20]:
def create_indexes():
    try:
        with engine.connect() as conn:
            print("A criar índices para otimização...")
            
            # 1. Índices nas Foreign Keys da Tabela de Factos
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_music_id ON fact_music(music_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_artist_id ON fact_music(artist_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_album_id ON fact_music(album_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_date_id ON fact_music(date_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_genre_id ON fact_music(genre_id);"))
            
            # 2. Índices de Performance (Para buscas frequentes)
            conn.execute(sqlalchemy.text("CREATE INDEX idx_music_popularity ON fact_music(music_popularity);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_review_score ON dim_review(classification);"))
            
            # 3. Se as dimensões foram criadas com to_sql(replace), elas não têm PKs definidas no SQL.
            conn.execute(sqlalchemy.text("ALTER TABLE dim_music ADD PRIMARY KEY (music_key);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_artist ADD PRIMARY KEY (artist_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_album ADD PRIMARY KEY (album_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_date ADD PRIMARY KEY (date_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_genre ADD PRIMARY KEY (genre_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_review ADD PRIMARY KEY (review_id);"))

            conn.commit() 
            print("Todos os índices e PKs foram criados com sucesso")

    except Exception as e:
        print(f"Erro ao criar índices: {e}")

create_indexes()

A criar índices para otimização...
Erro ao criar índices: (psycopg2.errors.UndefinedColumn) column "classification" does not exist

[SQL: CREATE INDEX idx_review_score ON dim_review(classification);]
(Background on this error at: https://sqlalche.me/e/20/f405)


---
# Phase 2: Information Retrieval System

## Step 4: Extract Searchable Documents

### Searchable Documents

In [21]:
fact_music = pd.read_csv('data/fact_music.tsv', sep='\t')
dim_music = pd.read_csv('data/dim_music.tsv', sep='\t')
dim_artist = pd.read_csv('data/dim_artist.tsv', sep='\t')
dim_album = pd.read_csv('data/dim_album.tsv', sep='\t')
dim_genre = pd.read_csv('data/dim_genre.tsv', sep='\t')
dim_date = pd.read_csv('data/dim_date.tsv', sep='\t')
dim_review = pd.read_csv('data/dim_review.tsv', sep='\t')

# Group reviews of the same album
reviews_grouped = dim_review.groupby('Album_ID').apply(
    lambda x: x[['Summary', 'Review']].rename(
        columns={'Summary': 'summary', 'Review': 'review'}
    ).to_dict('records')
).reset_index(name='reviews')

df = fact_music[['Music_ID', 'Artist_ID', 'Album_ID', 'Date_ID', 'Genre_ID', 'Word count']].copy()

# Add the dimensions to the df
df = df.merge(dim_music[['MUSIC_KEY', 'name', 'lyrics']], left_on='Music_ID', right_on='MUSIC_KEY', how='left')
df = df.merge(dim_artist[['Artist_ID', 'Name']], on='Artist_ID', how='left')
df = df.merge(dim_album[['Album_ID', 'Name']], on='Album_ID', how='left', suffixes=('', '_album'))
df = df.merge(dim_genre[['Genre_ID', 'Genre name']], on='Genre_ID', how='left')
df = df.merge(dim_date[['Date_ID', 'Complete Date']], on='Date_ID', how='left')
df = df.merge(reviews_grouped, on='Album_ID', how='left')

documents = []
for _, row in df.iterrows():
    doc = {
        "title": str(row['name']),
        "release_date": str(row['Complete Date']),
        "artist": {
            "name": str(row['Name'])
        },
        "album": {
            "name": str(row['Name_album'])
        },
        "genre": str(row['Genre name']),
        "content": {
            "lyrics": str(row['lyrics']),
        },
        "reviews": row['reviews']
    }
    documents.append(doc)

with open('jsons/searchable_documents.json', 'w', encoding='utf-8') as f:
    json.dump(documents, f, indent=2, ensure_ascii=False)

print(f"Sucesso! Foram gerados {len(documents)} documentos pesquisáveis.")

C:\Users\nunor\AppData\Local\Temp\ipykernel_9712\2307634019.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reviews_grouped = dim_review.groupby('Album_ID').apply(


Sucesso! Foram gerados 244632 documentos pesquisáveis.


### Measure Document

In [22]:
fact_music = pd.read_csv('data/fact_music.tsv', sep='\t')
dim_review = pd.read_csv('data/dim_review.tsv', sep='\t')

# Agrupamento das reviews por álbum (classificação)
review_scores = dim_review.groupby('Album_ID')['Classification'].mean().reset_index()
review_scores.rename(columns={'Classification': 'avg_review_score'}, inplace=True)

# Base: medidas numéricas da fact_music
df_m = fact_music[[
    'Music_ID', 'Album_ID',
    'Duration', 'Danceability', 'Energy', 'Loudness',
    'Speechiness', 'Acousticness', 'Instrumentalness',
    'Liveness', 'Music popularity', 'Word count'
]].copy()

df_m = df_m.merge(review_scores, on='Album_ID', how='left')

def safe_float(val, decimals=4):
    """Converte para float com precisão reduzida; retorna None se inválido."""
    try:
        return round(float(val), decimals) if pd.notnull(val) else None
    except (ValueError, TypeError):
        return None

def safe_int(val):
    try:
        return int(val) if pd.notnull(val) else None
    except (ValueError, TypeError):
        return None

measure_documents = []
for _, row in df_m.iterrows():
    doc = {
        "music_id": safe_int(row['Music_ID']),
        "measures": {
            "duration_ms":       safe_int(row['Duration']),
            "danceability":      safe_float(row['Danceability']),
            "energy":            safe_float(row['Energy']),
            "loudness":          safe_float(row['Loudness']),
            "speechiness":       safe_float(row['Speechiness']),
            "acousticness":      safe_float(row['Acousticness']),
            "instrumentalness":  safe_float(row['Instrumentalness']),
            "liveness":          safe_float(row['Liveness']),
            "music_popularity":  safe_int(row['Music popularity']),
            "word_count":        safe_int(row['Word count']),
            "avg_review_score":  safe_float(row['avg_review_score'])
        }
    }
    measure_documents.append(doc)

with open('jsons/measure_documents.json', 'w', encoding='utf-8') as f:
    json.dump(measure_documents, f, indent=2, ensure_ascii=False)

print(f"Sucesso! Foram gerados {len(measure_documents)} documentos de medidas.")


Sucesso! Foram gerados 243558 documentos de medidas.


## Step 5: Text Representation and Preprocessing

### Documents Text Representation

In [23]:
WEIGHTS = {
    'title':   5,
    'artist':  4,
    'lyrics':  3,
    'reviews': 2,
    'genre':   2,
}

def build_weighted_text(doc):
    parts = []

    parts += [doc.get('title', '')] * WEIGHTS['title']
    parts += [doc['artist'].get('name', '')] * WEIGHTS['artist']
    parts += [doc.get('genre', '')] * WEIGHTS['genre']

    lyrics = doc['content'].get('lyrics', '')
    parts += [lyrics] * WEIGHTS['lyrics']

    reviews = doc.get('reviews')
    if not isinstance(reviews, list):
        reviews = []

    review_text = ' '.join(
        r.get('summary', '') + ' ' + r.get('review', '')
        for r in reviews
        if isinstance(r, dict)
    )
    parts += [review_text] * WEIGHTS['reviews']

    return ' '.join(parts)

df = pd.DataFrame({
    "doc_id": range(len(documents)),
    "text": [build_weighted_text(doc) for doc in documents]
})

df.head()

,doc_id,text
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...
1,1,Starboy Starboy Starboy Starboy Starboy the we...
2,2,Starboy Starboy Starboy Starboy Starboy the we...
3,3,Mr. Brightside Mr. Brightside Mr. Brightside M...
4,4,Mr. Brightside Mr. Brightside Mr. Brightside M...


### Measures Text Representation

In [24]:
def extract_values(obj):
    values = []
    if isinstance(obj, dict):
        for v in obj.values():
            values.extend(extract_values(v))
    elif isinstance(obj, list):
        for item in obj:
            values.extend(extract_values(item))
    else:
        values.append(str(obj))
    return values

df_measures = pd.DataFrame({
    "measure_id": range(len(measure_documents)),
    "measures": [" ".join(extract_values(doc)) for doc in measure_documents]
})

df_measures.head()

,measure_id,measures
0,0,1 183956 0.464 0.417 -9.345 0.0256 0.136 0.022...
1,1,2 230467 0.682 0.592 -7.033 0.281 0.169 0.0 0....
2,2,3 230453 0.679 0.587 -7.015 0.276 0.141 0.0 0....
3,3,4 222973 0.352 0.911 -5.23 0.0747 0.0012 0.0 0...
4,4,5 222107 0.352 0.928 -3.71 0.0758 0.0011 0.0 0...


In [25]:
_CONTRACTIONS_PATH =  "jsons/contractions_en.json"
 
def _load_contractions(path: Path = _CONTRACTIONS_PATH) -> dict[str, str]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)
 

### Use only 1000 documents to speed up the process

In [26]:
df = df.iloc[:1000].copy()

df = df.reset_index(drop=True)

df_measures_sample = df_measures[df_measures['measure_id'].isin(df['doc_id'])]

### Clean text

In [27]:
def clean_text(s: str) -> str:
    # 1. Normalizar unicode (ex: é → e, ñ → n)
    s = unicodedata.normalize("NFKD", s)

    s = s.encode("ascii", "ignore").decode("ascii")

    s = s.lower()

    s = re.sub(r"[^a-z0-9\s'\-]", " ", s)  # preservar ' e -

    # Tornar múltiplos espaços em um único
    s = re.sub(r"\s+", " ", s)

    return s.strip()


df["text_after_processing"] = df["text"].apply(clean_text)
df.head()

,doc_id,text,text_after_processing
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...,i wanna be yours i wanna be yours i wanna be y...
1,1,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy the we...
2,2,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy the we...
3,3,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...
4,4,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...


### Normalize text

In [28]:
def normalize_text_fast(s: str, contractions: dict[str, str] | None = None) -> str:
    if contractions is None:
        contractions = _load_contractions()
    
    # 1. Criar um padrão único que engloba todas as contrações
    # O padrão será algo como: \b(can't|won't|it's)\b
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in contractions.keys()) + r')\b')
    
    # 2. Função interna que decide qual a expansão a usar
    def replace(match):
        return contractions[match.group(0)]
    
    # 3. Faz apenas UMA passagem pelo texto
    return pattern.sub(replace, s)

# Aplicar ao DataFrame
df["text_after_processing"] = df["text_after_processing"].apply(normalize_text_fast)

df.head()

,doc_id,text,text_after_processing
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...,i want to be yours i want to be yours i want t...
1,1,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy the we...
2,2,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy the we...
3,3,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...
4,4,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...


### Tokenization

In [29]:
def tokenize(s: str):
    return s.split() 

df["tokens"] = df["text_after_processing"].apply(tokenize)
df.head()

,doc_id,text,text_after_processing,tokens
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...,i want to be yours i want to be yours i want t...,"[i, want, to, be, yours, i, want, to, be, your..."
1,1,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy the we...,"[starboy, starboy, starboy, starboy, starboy, ..."
2,2,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy the we...,"[starboy, starboy, starboy, starboy, starboy, ..."
3,3,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...,"[mr, brightside, mr, brightside, mr, brightsid..."
4,4,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...,"[mr, brightside, mr, brightside, mr, brightsid..."


### Stop word removal

In [30]:
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stop_en = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_en]

df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)
df[["tokens", "tokens_no_stop"]].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nunor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,tokens,tokens_no_stop
0,"[i, want, to, be, yours, i, want, to, be, your...","[want, want, want, want, want, arctic, monkeys..."
1,"[starboy, starboy, starboy, starboy, starboy, ...","[starboy, starboy, starboy, starboy, starboy, ..."
2,"[starboy, starboy, starboy, starboy, starboy, ...","[starboy, starboy, starboy, starboy, starboy, ..."
3,"[mr, brightside, mr, brightside, mr, brightsid...","[mr, brightside, mr, brightside, mr, brightsid..."
4,"[mr, brightside, mr, brightside, mr, brightsid...","[mr, brightside, mr, brightside, mr, brightsid..."


### Stemming

In [31]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer() 

def apply_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

df["stems"] = df["tokens_no_stop"].apply(apply_stemming)
df[["tokens_no_stop", "stems"]].head()


,tokens_no_stop,stems
0,"[want, want, want, want, want, arctic, monkeys...","[want, want, want, want, want, arctic, monkey,..."
1,"[starboy, starboy, starboy, starboy, starboy, ...","[starboy, starboy, starboy, starboy, starboy, ..."
2,"[starboy, starboy, starboy, starboy, starboy, ...","[starboy, starboy, starboy, starboy, starboy, ..."
3,"[mr, brightside, mr, brightside, mr, brightsid...","[mr, brightsid, mr, brightsid, mr, brightsid, ..."
4,"[mr, brightside, mr, brightside, mr, brightsid...","[mr, brightsid, mr, brightsid, mr, brightsid, ..."


### Join Tokens

In [32]:
def join_tokens(tokens):
    return " ".join(tokens)  

df["text_no_stop"] = df["tokens_no_stop"].apply(join_tokens)  
df["text_stemming"] = df["stems"].apply(join_tokens)  

df[["text", "text_no_stop", "text_stemming"]].head()

,text,text_no_stop,text_stemming
0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...,want want want want want arctic monkeys arctic...,want want want want want arctic monkey arctic ...
1,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy weeknd...,starboy starboy starboy starboy starboy weeknd...
2,Starboy Starboy Starboy Starboy Starboy the we...,starboy starboy starboy starboy starboy weeknd...,starboy starboy starboy starboy starboy weeknd...
3,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...,mr brightsid mr brightsid mr brightsid mr brig...
4,Mr. Brightside Mr. Brightside Mr. Brightside M...,mr brightside mr brightside mr brightside mr b...,mr brightsid mr brightsid mr brightsid mr brig...


### Vector Representation — Searchable Documents (TF-IDF)

In [33]:
import pickle
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df["text_stemming"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulário: {len(tfidf_vectorizer.vocabulary_)} termos")

scipy.sparse.save_npz('jsons/tfidf_matrix.npz', tfidf_matrix)

with open('jsons/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

TF-IDF matrix shape: (1000, 28446)
Vocabulário: 28446 termos


### Vector Representation — Measure Documents

In [34]:
import numpy as np

METRIC_KEYS = [
    'Duration', 'Danceability', 'Energy', 'Loudness',
    'Speechiness', 'Acousticness', 'Instrumentalness',
    'Liveness', 'Music popularity', 'Word count', 'avg_review_score'
]

raw = df_m[METRIC_KEYS].fillna(0).values.astype(float)

def zscore_col(col):
    mu, sigma = col.mean(), col.std()
    return (col - mu) / sigma if sigma > 0 else np.zeros_like(col)

measure_matrix = np.zeros_like(raw)
col = {k: i for i, k in enumerate(METRIC_KEYS)}

for k in ['Danceability', 'Energy', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Liveness']:
    measure_matrix[:, col[k]] = raw[:, col[k]]

measure_matrix[:, col['Loudness']]        = zscore_col(raw[:, col['Loudness']])
measure_matrix[:, col['Duration']]        = zscore_col(raw[:, col['Duration']] / 60000)
measure_matrix[:, col['Music popularity']]= raw[:, col['Music popularity']] / 100
measure_matrix[:, col['Word count']]      = zscore_col(np.log1p(raw[:, col['Word count']]))
measure_matrix[:, col['avg_review_score']]= raw[:, col['avg_review_score']] / 10

print(f"Measure matrix shape: {measure_matrix.shape}")
np.save('jsons/measure_matrix.npy', measure_matrix)

Measure matrix shape: (243558, 11)


In [35]:
df_m = df_m[df_m['Music_ID'].isin(df['doc_id'])].reset_index(drop=True)
measure_matrix = measure_matrix[df_m.index]

## Step 6: Implement Retrieval Models

### Boolean Retrieval

In [36]:
def boolean_search_AND(query: str):
    text = clean_text(query)       
    text = normalize_text_fast(text)   
    text = tokenize(text)          
    text = remove_stopwords(text)  
    terms = apply_stemming(text) 

    print(terms)
  

    results = []
    for _, row in df.iterrows():
        if all(t in row["stems"] for t in terms):
            results.append(row)

    if not results:
        return pd.DataFrame(columns=["doc_id", "text"])

    return pd.DataFrame(results)[["doc_id", "text"]]


boolean_search_AND("I Wanna Be Yours tonight my love I'm in love")

['want', 'tonight', 'love', 'love']


,doc_id,text
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...
12,12,Do I Wanna Know? Do I Wanna Know? Do I Wanna K...
13,13,No. 1 Party Anthem No. 1 Party Anthem No. 1 Pa...
18,18,Fade Into You Fade Into You Fade Into You Fade...
24,24,Why'd You Only Call Me When You're High? Why'd...
...,...,...
960,960,We're Going to Be Friends We're Going to Be Fr...
966,966,One For The Road One For The Road One For The ...
977,977,You Got Me You Got Me You Got Me You Got Me Yo...
997,997,Impatient Impatient Impatient Impatient Impati...


In [37]:
def boolean_search_OR(query: str):
    text = clean_text(query)
    text = normalize_text_fast(text)
    text = tokenize(text)
    text = remove_stopwords(text)
    terms = apply_stemming(text)

    print(terms)


    results = []
    for _, row in df.iterrows():
        if any(t in row["stems"] for t in terms):
            results.append(row)

    if not results:
        return pd.DataFrame(columns=["doc_id", "text"])

    return pd.DataFrame(results)[["doc_id", "text"]]

boolean_search_OR("I Wanna Be Yours tonight my love I'm in love")

['want', 'tonight', 'love', 'love']


,doc_id,text
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...
1,1,Starboy Starboy Starboy Starboy Starboy the we...
2,2,Starboy Starboy Starboy Starboy Starboy the we...
3,3,Mr. Brightside Mr. Brightside Mr. Brightside M...
4,4,Mr. Brightside Mr. Brightside Mr. Brightside M...
...,...,...
995,995,Never Be Like You Never Be Like You Never Be L...
996,996,Never Be Like You Never Be Like You Never Be L...
997,997,Impatient Impatient Impatient Impatient Impati...
998,998,One Take One Take One Take One Take One Take l...


### Vector Space Model (TF-IDF)

In [38]:
def tfidf_search(query: str, top_k=5):
    text = clean_text(query)
    text = normalize_text_fast(text) 
    text = tokenize(text)
    text = remove_stopwords(text)
    terms = apply_stemming(text)
   
    q_vec = tfidf_vectorizer.transform([" ".join(terms)])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    
    # Obtém as posições (offsets) dos top_k resultados
    idx = sims.argsort()[::-1][:top_k]
    
    # USAR .iloc para selecionar pelas posições físicas (0, 1, 2...)
    return df.iloc[idx][["doc_id", "text"]].assign(score=sims[idx])

# Testar novamente
tfidf_search("I Wanna Be Yours", top_k=5)

,doc_id,text,score
255,255,Don't You Want Me Don't You Want Me Don't You ...,0.628066
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...,0.485920
474,474,End Game End Game End Game End Game End Game t...,0.308765
339,339,1-800-273-8255 1-800-273-8255 1-800-273-8255 1...,0.307994
60,60,Solo Solo Solo Solo Solo future future future ...,0.246354


### BM25

In [39]:
bm25 = BM25Okapi(df["stems"].tolist())

def bm25_search(query: str, top_k=5):
    text = clean_text(query)
    text = normalize_text_fast(text)
    text = tokenize(text)
    text = remove_stopwords(text)
    terms = apply_stemming(text)

    scores = bm25.get_scores(terms)
    
    # Obtém as POSIÇÕES dos maiores scores
    idx = np.argsort(scores)[::-1][:top_k]
    
    # .iloc seleciona por posição (0, 1, 2...), resolvendo o KeyError
    return df.iloc[idx][["doc_id", "text"]].assign(score=scores[idx])

bm25_search("I Wanna Be Yours", top_k=5)

,doc_id,text,score
255,255,Don't You Want Me Don't You Want Me Don't You ...,3.265359
0,0,I Wanna Be Yours I Wanna Be Yours I Wanna Be Y...,3.257092
771,771,Make It Wit Chu Make It Wit Chu Make It Wit Ch...,3.242978
772,772,Make It Wit Chu Make It Wit Chu Make It Wit Ch...,3.242978
669,669,Selfish Selfish Selfish Selfish Selfish pnb ro...,3.241461


### Compare results

In [40]:
def compare(query: str, top_k=5):
    print(f"QUERY: {query}\n")

    print("=== Boolean (AND) ===")
    display(boolean_search_AND(query))

    print("=== Boolean (OR) ===")
    display(boolean_search_OR(query))

    print("\n=== Vectorial (TF-IDF) ===")
    display(tfidf_search(query, top_k=top_k))

    print("\n=== BM25 ===")
    display(bm25_search(query, top_k=top_k))

compare("drake toronto views", top_k=5)

QUERY: drake toronto views

=== Boolean (AND) ===
['drake', 'toronto', 'view']


,doc_id,text
6,6,One Dance One Dance One Dance One Dance One Da...
7,7,One Dance One Dance One Dance One Dance One Da...
8,8,One Dance One Dance One Dance One Dance One Da...
9,9,One Dance One Dance One Dance One Dance One Da...
127,127,9 9 9 9 9 drake drake drake drake hip-hop hip-...
148,148,Hotline Bling Hotline Bling Hotline Bling Hotl...
149,149,Hotline Bling Hotline Bling Hotline Bling Hotl...
150,150,Hotline Bling Hotline Bling Hotline Bling Hotl...
316,316,Feel No Ways Feel No Ways Feel No Ways Feel No...
427,427,TSU TSU TSU TSU TSU drake drake drake drake hi...


=== Boolean (OR) ===
['drake', 'toronto', 'view']


,doc_id,text
6,6,One Dance One Dance One Dance One Dance One Da...
7,7,One Dance One Dance One Dance One Dance One Da...
8,8,One Dance One Dance One Dance One Dance One Da...
9,9,One Dance One Dance One Dance One Dance One Da...
15,15,No Role Modelz No Role Modelz No Role Modelz N...
...,...,...
986,986,Weston Road Flows Weston Road Flows Weston Roa...
987,987,Boo'd Up Boo'd Up Boo'd Up Boo'd Up Boo'd Up e...
990,990,Deja Vu Deja Vu Deja Vu Deja Vu Deja Vu j. col...
991,991,Gas Pedal Gas Pedal Gas Pedal Gas Pedal Gas Pe...



=== Vectorial (TF-IDF) ===


,doc_id,text,score
561,561,Summers Over Interlude Summers Over Interlude ...,0.409421
316,316,Feel No Ways Feel No Ways Feel No Ways Feel No...,0.346825
165,165,Yebba’s Heartbreak Yebba’s Heartbreak Yebba’s ...,0.344486
736,736,Get Along Better Get Along Better Get Along Be...,0.306204
8,8,One Dance One Dance One Dance One Dance One Da...,0.302080



=== BM25 ===


,doc_id,text,score
127,127,9 9 9 9 9 drake drake drake drake hip-hop hip-...,14.123913
561,561,Summers Over Interlude Summers Over Interlude ...,13.463717
316,316,Feel No Ways Feel No Ways Feel No Ways Feel No...,13.147306
444,444,Fire & Desire Fire & Desire Fire & Desire Fire...,13.102929
9,9,One Dance One Dance One Dance One Dance One Da...,12.978078


## Step 7: Evaluation of the Retrieval System

### Select 20 random documents using a seed

In [41]:
random.seed(42)

sampled_indices = random.sample(range(len(df)), min(20, len(df)))
documents_20 = df.iloc[sampled_indices].reset_index(drop=True)

print(f"Documentos selecionados: {len(documents_20)}")
print(documents_20[['doc_id', 'text']])

# Guardar resultado em txt
output_lines = []
output_lines.append(f"Documentos selecionados: {len(documents_20)}\n\n")

for _, row in documents_20.iterrows():
    line = f"{row['doc_id']} {row['text']}\n"
    print(line)
    output_lines.append(line)

with open("documents_20.txt", "w", encoding="utf-8") as f:
    f.writelines(output_lines)

print("Ficheiro guardado: documents_20.txt")

Documentos selecionados: 20
    doc_id                                               text
0      654  Brain Damage Brain Damage Brain Damage Brain D...
1      114  Just A Lil Bit Just A Lil Bit Just A Lil Bit J...
2       25  Empire State Of Mind Empire State Of Mind Empi...
3      759  Blind Blind Blind Blind Blind sza sza sza sza ...
4      281  Young Folks Young Folks Young Folks Young Folk...
5      250  When You Were Young When You Were Young When Y...
6      228  Little Dark Age Little Dark Age Little Dark Ag...
7      142  Ayo Technology Ayo Technology Ayo Technology A...
8      754  A Pearl A Pearl A Pearl A Pearl A Pearl mitski...
9      104  I Wonder I Wonder I Wonder I Wonder I Wonder k...
10     692  You're Gonna Live Forever in Me You're Gonna L...
11     758  goodnight n go goodnight n go goodnight n go g...
12     913  The Adventures of Rain Dance Maggie The Advent...
13     558  Walk It Talk It Walk It Talk It Walk It Talk I...
14      89  New Person, Same Old Mistakes 

### Define 20 Test Queries

*Queries generated having in consideration the sampled documents.*

In [53]:
import pandas as pd

queries = [
    # Queries diretas por título/álbum/artista — devem ter resultados óbvios
    {"qid": 1, "query": "dark side moon lunatic"},
    {"qid": 2, "query": "young style old folks"},

    # Queries por tema lírico — testam a qualidade do IR nas letras
    {"qid": 3, "query": "new york city dreams street"},
    {"qid": 4, "query": "love stay night hurt"},

    # Queries por género/sonoridade — testam o campo genre + reviews
    {"qid": 5, "query": "hip hop beats rap"},
    {"qid": 6, "query": "rock guitar pop band"},

    # Queries baseadas em conteúdo das reviews/letras específicas
    {"qid": 7, "query": "technology hypnotized touch"},
    {"qid": 8, "query": "mistakes person changes"},

    # Queries ambíguas — testam robustez dos modelos
    {"qid": 9, "query": "club dance floor money"},
    {"qid": 10, "query": "dreams wonder star"},

    # Queries adicionais para melhorar o treino do LTR
    {"qid": 11, "query": "indie alternative psychedelic vibes"},
    {"qid": 12, "query": "emotional sad ballad heartbreak"},
    {"qid": 13, "query": "classic rock guitar anthem"},
    {"qid": 14, "query": "new york city rap anthem"},
    {"qid": 15, "query": "psychedelic space rock dark"},
    {"qid": 16, "query": "trap street rap mumble"},
    {"qid": 17, "query": "r&b late night smooth"},
    {"qid": 18, "query": "conscious rap clever wordplay"},
    {"qid": 19, "query": "dance floor banger club"},
    {"qid": 20, "query": "heartbreak sad pop love"},
]

df_q = pd.DataFrame(queries)
df_q

,qid,query
0,1,dark side moon lunatic
1,2,young style old folks
2,3,new york city dreams street
3,4,love stay night hurt
4,5,hip hop beats rap
5,6,rock guitar pop band
6,7,technology hypnotized touch
7,8,mistakes person changes
8,9,club dance floor money
9,10,dreams wonder star


### Relevance Judgments (Qrels)

*Relevance values generated by AI*

In [54]:
import pandas as pd

qrels = [
    # dark side moon lunatic → Brain Damage (Pink Floyd)
    {"qid": 1, "doc_id": 654, "rel": 3},

    # young style old folks → Young Folks, When You Were Young
    {"qid": 2, "doc_id": 281, "rel": 3},
    {"qid": 2, "doc_id": 250, "rel": 2},

    # new york city dreams street → Empire State Of Mind
    {"qid": 3, "doc_id": 25, "rel": 3},
    {"qid": 3, "doc_id": 104, "rel": 1},

    # love stay night hurt → Stay With Me, Goodnight N Go, A Pearl
    {"qid": 4, "doc_id": 95, "rel": 3},
    {"qid": 4, "doc_id": 758, "rel": 2},
    {"qid": 4, "doc_id": 754, "rel": 1},

    # hip hop beats rap → HUMBLE., Ayo Technology, Just A Lil Bit
    {"qid": 5, "doc_id": 32, "rel": 3},
    {"qid": 5, "doc_id": 142, "rel": 3},
    {"qid": 5, "doc_id": 114, "rel": 2},
    {"qid": 5, "doc_id": 604, "rel": 1},

    # rock guitar pop band → Rain Dance Maggie, When You Were Young, Take Me Out
    {"qid": 6, "doc_id": 913, "rel": 3},
    {"qid": 6, "doc_id": 250, "rel": 2},
    {"qid": 6, "doc_id": 30, "rel": 2},
    {"qid": 6, "doc_id": 228, "rel": 1},

    # technology hypnotized touch → Ayo Technology
    {"qid": 7, "doc_id": 142, "rel": 3},
    {"qid": 7, "doc_id": 114, "rel": 1},

    # mistakes person changes → New Person Same Old Mistakes
    {"qid": 8, "doc_id": 89, "rel": 3},
    {"qid": 8, "doc_id": 754, "rel": 1},

    # club dance floor money → Did It Again, Just A Lil Bit, Walk It Talk It
    {"qid": 9, "doc_id": 604, "rel": 3},
    {"qid": 9, "doc_id": 114, "rel": 2},
    {"qid": 9, "doc_id": 558, "rel": 2},
    {"qid": 9, "doc_id": 142, "rel": 1},

    # dreams wonder star → I Wonder, Little Dark Age, Brain Damage
    {"qid": 10, "doc_id": 104, "rel": 3},
    {"qid": 10, "doc_id": 228, "rel": 2},
    {"qid": 10, "doc_id": 654, "rel": 1},

    # indie alternative psychedelic vibes
    {"qid": 11, "doc_id": 89,  "rel": 3},
    {"qid": 11, "doc_id": 228, "rel": 2},
    {"qid": 11, "doc_id": 281, "rel": 1},

    # emotional sad ballad heartbreak
    {"qid": 12, "doc_id": 95,  "rel": 3},
    {"qid": 12, "doc_id": 754, "rel": 2},
    {"qid": 12, "doc_id": 758, "rel": 1},

    # classic rock guitar anthem
    {"qid": 13, "doc_id": 913, "rel": 3},
    {"qid": 13, "doc_id": 250, "rel": 2},
    {"qid": 13, "doc_id": 30,  "rel": 1},

    # new york city rap anthem
    {"qid": 14, "doc_id": 25,  "rel": 3},
    {"qid": 14, "doc_id": 104, "rel": 1},

    # psychedelic space rock dark
    {"qid": 15, "doc_id": 654, "rel": 3},
    {"qid": 15, "doc_id": 228, "rel": 2},
    {"qid": 15, "doc_id": 89,  "rel": 1},

    # trap street rap mumble
    {"qid": 16, "doc_id": 604, "rel": 3},
    {"qid": 16, "doc_id": 432, "rel": 2},
    {"qid": 16, "doc_id": 114, "rel": 1},

    # r&b late night smooth
    {"qid": 17, "doc_id": 759, "rel": 3},
    {"qid": 17, "doc_id": 758, "rel": 2},
    {"qid": 17, "doc_id": 692, "rel": 1},

    # kendrick conscious rap
    {"qid": 18, "doc_id": 32,  "rel": 3},
    {"qid": 18, "doc_id": 104, "rel": 2},
    {"qid": 18, "doc_id": 142, "rel": 1},

    # dance floor banger club
    {"qid": 19, "doc_id": 558, "rel": 3},
    {"qid": 19, "doc_id": 604, "rel": 2},
    {"qid": 19, "doc_id": 142, "rel": 1},

    # heartbreak sad pop love
    {"qid": 20, "doc_id": 95,  "rel": 3},
    {"qid": 20, "doc_id": 759, "rel": 2},
    {"qid": 20, "doc_id": 754, "rel": 1},
]

df_qrels = pd.DataFrame(qrels)
df_qrels

,qid,doc_id,rel
0,1,654,3
1,2,281,3
2,2,250,2
3,3,25,3
4,3,104,1
5,4,95,3
6,4,758,2
7,4,754,1
8,5,32,3
9,5,142,3


### Ranking Functions

In [55]:
def rank_tfidf(query: str, top_k=None):
    top_k = top_k or len(df)
    text = clean_text(query)
    text = normalize_text_fast(text)
    text = tokenize(text)
    text = remove_stopwords(text)
    terms = apply_stemming(text)

    q_vec = tfidf_vectorizer.transform([" ".join(terms)])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()

    idx = sims.argsort()[::-1][:top_k]
    ranking = idx.tolist()          # índice posicional, não doc_id
    scores = sims[idx].tolist()
    return ranking, scores

def rank_bm25(query: str, top_k=None):
    top_k = top_k or len(df)
    text = clean_text(query)
    text = normalize_text_fast(text)
    text = tokenize(text)
    text = remove_stopwords(text)
    terms = apply_stemming(text)

    scores_arr = bm25.get_scores(terms)
    idx = np.argsort(scores_arr)[::-1][:top_k]
    ranking = idx.tolist()          # índice posicional, não doc_id
    scores = scores_arr[idx].tolist()
    return ranking, scores

### Evaluation Metrics

In [56]:
def rel_dict_for_qid(df_qrels, qid):
    sub = df_qrels[df_qrels["qid"] == qid]  # filter rows matching the query id
    return {int(r.doc_id): int(r.rel) for r in sub.itertuples(index=False)}  # map doc_id → relevance score

def precision_at_k(ranking, relevant_set, k): #Is what the system retrieved relevant?
    topk = ranking[:k]  # select top-k retrieved documents
    hits = sum(1 for d in topk if d in relevant_set)  # count relevant documents in top-k
    return hits / k if k > 0 else 0.0  # precision = relevant retrieved / k

def recall_at_k(ranking, relevant_set, k): #Did it find all relevant documents?
    topk = ranking[:k]  # select top-k retrieved documents
    hits = sum(1 for d in topk if d in relevant_set)  # count relevant documents found
    return hits / len(relevant_set) if len(relevant_set) > 0 else 0.0  # recall = relevant retrieved / total relevant

def f1(p, r): #Balance between precision and recall
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0  # harmonic mean of precision and recall

def average_precision(ranking, relevant_set): #Are relevant documents ranked higher?
    # AP using binary relevance (rel > 0)
    hit_count = 0  # number of relevant documents found so far
    sum_prec = 0.0  # accumulated precision values
    for i, d in enumerate(ranking, start=1):  # iterate over ranking with position index
        if d in relevant_set:  # check if document is relevant
            hit_count += 1  # increment relevant count
            sum_prec += hit_count / i  # add precision at current rank
    return sum_prec / len(relevant_set) if len(relevant_set) > 0 else 0.0  # normalize by total relevant docs

def dcg_at_k(ranking, rel_by_doc, k):
    dcg = 0.0  # initialize discounted cumulative gain
    for i, d in enumerate(ranking[:k], start=1):  # iterate through top-k results
        rel = rel_by_doc.get(d, 0)  # get relevance score (default 0)
        dcg += (2**rel - 1) / math.log2(i + 1)  # compute DCG with log discount
    return dcg  # return DCG value

def ndcg_at_k(ranking, rel_by_doc, k):
    ideal = sorted(rel_by_doc.keys(), key=lambda d: rel_by_doc.get(d, 0), reverse=True)  # sort docs by relevance (ideal ranking)
    # optionally extend ideal ranking with non-relevant docs (not required here)
    idcg = dcg_at_k(ideal, rel_by_doc, k)  # compute ideal DCG
    return dcg_at_k(ranking, rel_by_doc, k) / idcg if idcg > 0 else 0.0  # normalize DCG by ideal DCG

### Evaluation Metrics

In [57]:
def evaluate_model(model_name, rank_fn, k=5):
    rows = []  # store per-query evaluation results
    aps = []  # store average precision values for all queries
    ndcgs = []  # store nDCG values for all queries
    ps = []  # store precision values
    rs = []  # store recall values
    f1s = []  # store F1 values

    for q in df_q.itertuples(index=False):
        qid, qtext = int(q.qid), q.query  # extract query id and query text
        rel_by_doc = rel_dict_for_qid(df_qrels, qid)  # get relevance scores per document
        relevant_set = {d for d, rel in rel_by_doc.items() if rel > 0}  # set of relevant documents

        ranking, scores = rank_fn(qtext)  # generate ranked list of documents and their scores

        p = precision_at_k(ranking, relevant_set, k)  # compute precision@k
        r = recall_at_k(ranking, relevant_set, k)  # compute recall@k
        f = f1(p, r)  # compute F1-score
        ap = average_precision(ranking, relevant_set)  # compute average precision
        ndcg = ndcg_at_k(ranking, rel_by_doc, k)  # compute normalized DCG@k

        aps.append(ap)  # store AP for averaging
        ndcgs.append(ndcg)  # store nDCG for averaging
        ps.append(p)  # store precision
        rs.append(r)  # store recall
        f1s.append(f)  # store F1

        rows.append({
            "modelo": model_name,  # model name
            "qid": qid,  # query id
            "query": qtext,  # query text
            f"P@{k}": p,  # precision at k
            f"R@{k}": r,  # recall at k
            f"F1@{k}": f,  # F1-score at k
            "AP": ap,  # average precision
            f"nDCG@{k}": ndcg  # normalized DCG at k
        })

    df_res = pd.DataFrame(rows)  # create dataframe with per-query results
    summary = pd.DataFrame([{
        "modelo": model_name,  # model name
        f"m_P@{k}": float(np.mean(ps)),  # mean precision@k
        f"m_R@{k}": float(np.mean(rs)),  # mean recall@k
        f"m_F1@{k}": float(np.mean(f1s)),  # mean F1@k
        "MAP": float(np.mean(aps)),  # mean average precision
        f"m_nDCG@{k}": float(np.mean(ndcgs))  # mean nDCG@k
    }])

    return df_res, summary  # return detailed results and summary metrics

k = 5  # evaluation cutoff
res_tfidf, sum_tfidf = evaluate_model("TF-IDF", rank_tfidf, k=k)  # evaluate TF-IDF model
res_bm25,  sum_bm25  = evaluate_model("BM25",  rank_bm25,  k=k)  # evaluate BM25 model

display(res_tfidf)  # display TF-IDF detailed results
display(res_bm25)  # display BM25 detailed results

display(pd.concat([sum_tfidf, sum_bm25], ignore_index=True))  # display combined summary results

,modelo,qid,query,P@5,R@5,F1@5,AP,nDCG@5
0,TF-IDF,1,dark side moon lunatic,0.2,1.000000,0.333333,1.000000,1.000000
1,TF-IDF,2,young style old folks,0.2,0.500000,0.285714,0.392857,0.496639
2,TF-IDF,3,new york city dreams street,0.2,0.500000,0.285714,0.521277,0.917319
3,TF-IDF,4,love stay night hurt,0.2,0.333333,0.250000,0.179456,0.470202
4,TF-IDF,5,hip hop beats rap,0.0,0.000000,0.000000,0.017202,0.000000
5,TF-IDF,6,rock guitar pop band,0.0,0.000000,0.000000,0.022445,0.000000
6,TF-IDF,7,technology hypnotized touch,0.2,0.500000,0.285714,0.254444,0.578764
7,TF-IDF,8,mistakes person changes,0.2,0.500000,0.285714,0.501337,0.917319
8,TF-IDF,9,club dance floor money,0.0,0.000000,0.000000,0.020736,0.000000
9,TF-IDF,10,dreams wonder star,0.2,0.333333,0.250000,0.169932,0.470202


,modelo,qid,query,P@5,R@5,F1@5,AP,nDCG@5
0,BM25,1,dark side moon lunatic,0.2,1.000000,0.333333,1.000000,1.000000
1,BM25,2,young style old folks,0.2,0.500000,0.285714,0.255747,0.496639
2,BM25,3,new york city dreams street,0.2,0.500000,0.285714,0.502786,0.917319
3,BM25,4,love stay night hurt,0.2,0.333333,0.250000,0.180063,0.470202
4,BM25,5,hip hop beats rap,0.0,0.000000,0.000000,0.045660,0.000000
5,BM25,6,rock guitar pop band,0.0,0.000000,0.000000,0.006754,0.000000
6,BM25,7,technology hypnotized touch,0.2,0.500000,0.285714,0.505376,0.917319
7,BM25,8,mistakes person changes,0.2,0.500000,0.285714,0.501339,0.917319
8,BM25,9,club dance floor money,0.2,0.250000,0.222222,0.056805,0.035742
9,BM25,10,dreams wonder star,0.2,0.333333,0.250000,0.336041,0.745253


,modelo,m_P@5,m_R@5,m_F1@5,MAP,m_nDCG@5
0,TF-IDF,0.09,0.2250,0.125595,0.195983,0.296373
1,BM25,0.10,0.2375,0.136706,0.216430,0.344366


## Step 8: Learning to Rank

A Learning-to-Rank model (Gradient Boosting Regressor) is trained using TF-IDF and BM25 scores combined with the numeric features from the measure matrix as input features, and the relevance judgments as labels.

In [58]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

doc_id_to_measure_idx = {
    int(df_m.iloc[i]['Music_ID']): i 
    for i in range(len(df_m))
}

METRIC_KEYS = [
    'Duration', 'Danceability', 'Energy', 'Loudness',
    'Speechiness', 'Acousticness', 'Instrumentalness',
    'Liveness', 'Music popularity', 'Word count', 'avg_review_score'
]

# Construir dataset de treino
# Para cada query, para cada documento no df, calcular features
rows = []

for q in df_q.itertuples(index=False):
    qid, qtext = int(q.qid), q.query
    rel_by_doc = rel_dict_for_qid(df_qrels, qid)

    # Scores TF-IDF e BM25 para todos os docs
    ranking_tfidf, scores_tfidf = rank_tfidf(qtext)
    ranking_bm25, scores_bm25   = rank_bm25(qtext)

    # Criar dicionários posição → score
    tfidf_score = {pos: s for pos, s in zip(ranking_tfidf, scores_tfidf)}
    bm25_score  = {pos: s for pos, s in zip(ranking_bm25, scores_bm25)}

    for pos_idx, row in df.iterrows():
        doc_id = int(row['doc_id'])
        rel = rel_by_doc.get(doc_id, 0)

        # Features de texto
        tf_s  = tfidf_score.get(pos_idx, 0.0)
        bm_s  = bm25_score.get(pos_idx, 0.0)

        # Features numéricas da measure_matrix
        m_idx = doc_id_to_measure_idx.get(doc_id, None)
        if m_idx is not None:
            measure_feats = measure_matrix[m_idx].tolist()
        else:
            measure_feats = [0.0] * len(METRIC_KEYS)

        rows.append({
            'qid':    qid,
            'doc_id': doc_id,
            'rel':    rel,
            'tfidf':  tf_s,
            'bm25':   bm_s,
            **{k: measure_feats[i] for i, k in enumerate(METRIC_KEYS)}
        })

df_train = pd.DataFrame(rows)

# Features e label
feature_cols = ['tfidf', 'bm25'] + METRIC_KEYS
X = df_train[feature_cols].values
y = df_train['rel'].values

# Treinar modelo
ltr_model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
ltr_model.fit(X, y)

print("Modelo treinado!")
print(f"Features: {feature_cols}")

# Feature importance
importances = pd.Series(ltr_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importance:")
print(importances)

Modelo treinado!
Features: ['tfidf', 'bm25', 'Duration', 'Danceability', 'Energy', 'Loudness', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Liveness', 'Music popularity', 'Word count', 'avg_review_score']

Feature Importance:
tfidf               0.360552
bm25                0.205011
Acousticness        0.075777
Instrumentalness    0.069908
Liveness            0.055243
Loudness            0.050089
avg_review_score    0.042534
Speechiness         0.035246
Duration            0.035008
Energy              0.026339
Danceability        0.021367
Music popularity    0.016639
Word count          0.006287
dtype: float64


### LTR Ranking Function

In [59]:
def rank_ltr(query: str, top_k=None):
    top_k = top_k or len(df)

    ranking_tfidf, scores_tfidf = rank_tfidf(query)
    ranking_bm25, scores_bm25   = rank_bm25(query)

    tfidf_score = {pos: s for pos, s in zip(ranking_tfidf, scores_tfidf)}
    bm25_score  = {pos: s for pos, s in zip(ranking_bm25, scores_bm25)}

    rows = []
    for pos_idx, row in df.iterrows():
        doc_id = int(row['doc_id'])
        tf_s  = tfidf_score.get(pos_idx, 0.0)
        bm_s  = bm25_score.get(pos_idx, 0.0)

        m_idx = doc_id_to_measure_idx.get(doc_id, None)
        if m_idx is not None:
            measure_feats = measure_matrix[m_idx].tolist()
        else:
            measure_feats = [0.0] * len(METRIC_KEYS)

        rows.append([tf_s, bm_s] + measure_feats)

    X_q = np.array(rows)
    scores = ltr_model.predict(X_q)

    idx = np.argsort(scores)[::-1][:top_k]
    return idx.tolist(), scores[idx].tolist()

### Evaluate LTR & Compare All Models

In [60]:
res_ltr, sum_ltr = evaluate_model("LTR", rank_ltr, k=5)

# Resultados por query — todos os modelos
display(pd.concat([res_tfidf, res_bm25, res_ltr], ignore_index=True))

# Resumo final — todos os modelos
display(pd.concat([sum_tfidf, sum_bm25, sum_ltr], ignore_index=True))

,modelo,qid,query,P@5,R@5,F1@5,AP,nDCG@5
0,TF-IDF,1,dark side moon lunatic,0.2,1.000000,0.333333,1.000000,1.000000
1,TF-IDF,2,young style old folks,0.2,0.500000,0.285714,0.392857,0.496639
2,TF-IDF,3,new york city dreams street,0.2,0.500000,0.285714,0.521277,0.917319
3,TF-IDF,4,love stay night hurt,0.2,0.333333,0.250000,0.179456,0.470202
4,TF-IDF,5,hip hop beats rap,0.0,0.000000,0.000000,0.017202,0.000000
5,TF-IDF,6,rock guitar pop band,0.0,0.000000,0.000000,0.022445,0.000000
6,TF-IDF,7,technology hypnotized touch,0.2,0.500000,0.285714,0.254444,0.578764
7,TF-IDF,8,mistakes person changes,0.2,0.500000,0.285714,0.501337,0.917319
8,TF-IDF,9,club dance floor money,0.0,0.000000,0.000000,0.020736,0.000000
9,TF-IDF,10,dreams wonder star,0.2,0.333333,0.250000,0.169932,0.470202


,modelo,m_P@5,m_R@5,m_F1@5,MAP,m_nDCG@5
0,TF-IDF,0.09,0.2250,0.125595,0.195983,0.296373
1,BM25,0.10,0.2375,0.136706,0.216430,0.344366
2,LTR,0.38,0.6750,0.475992,0.708051,0.787345


## Bonus: Retrieval-Augmented Generation (RAG)

The system uses the LTR model to retrieve relevant documents from the warehouse-derived index, then passes them to a Large Language Model (Groq/Llama) to generate a natural language answer.

In [62]:
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Load measure_documents
with open('jsons/measure_documents.json', encoding='utf-8') as f:
    measure_docs = json.load(f)

df_measure_docs = pd.DataFrame([
    {"music_id": doc["music_id"], **doc["measures"]}
    for doc in measure_docs
])

def extract_filters(query: str) -> dict:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """Extract numeric filters from a music query. Return ONLY a JSON object.

Available fields (all normalized 0-1):
- music_popularity (e.g. "popular" > 0.7)
- danceability (e.g. "danceable" > 0.7)
- energy (e.g. "energetic" > 0.7, "calm" < 0.4)
- acousticness (e.g. "acoustic" > 0.7)
- instrumentalness (e.g. "instrumental" > 0.5)
- liveness (e.g. "live" > 0.7)
- speechiness (e.g. "rap/speech" > 0.5)
- avg_review_score (e.g. "well reviewed" > 0.7)
- duration_ms (e.g. "short" < 0.3, "long" > 0.7)

Use "gt" for > and "lt" for <. Example: {"music_popularity": {"gt": 0.7}}
If no filters, return {}."""
            },
            {"role": "user", "content": query}
        ],
        max_tokens=150
    )
    raw = re.sub(r'```json|```', '', response.choices[0].message.content).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}

def apply_metric_filters(filters: dict) -> list:
    mask = pd.Series([True] * len(df_measure_docs), index=df_measure_docs.index)
    for field, condition in filters.items():
        if field not in df_measure_docs.columns:
            continue
        if 'gt' in condition:
            mask &= df_measure_docs[field] > condition['gt']
        if 'lt' in condition:
            mask &= df_measure_docs[field] < condition['lt']
    return df_measure_docs[mask]['music_id'].tolist()

def rank_ltr_filtered(query: str, valid_ids: list, top_k=5):
    df_sub = df[df['doc_id'].isin(valid_ids)].reset_index(drop=True)

    ranking_tfidf, scores_tfidf = rank_tfidf(query)
    ranking_bm25, scores_bm25   = rank_bm25(query)

    tfidf_score = {pos: s for pos, s in zip(ranking_tfidf, scores_tfidf)}
    bm25_score  = {pos: s for pos, s in zip(ranking_bm25, scores_bm25)}

    rows = []
    for pos_idx, row in df_sub.iterrows():
        doc_id = int(row['doc_id'])
        original_pos = df[df['doc_id'] == doc_id].index[0]

        tf_s = tfidf_score.get(original_pos, 0.0)
        bm_s = bm25_score.get(original_pos, 0.0)

        m_idx = doc_id_to_measure_idx.get(doc_id, None)
        measure_feats = measure_matrix[m_idx].tolist() if m_idx is not None else [0.0] * len(METRIC_KEYS)

        rows.append([tf_s, bm_s] + measure_feats)

    X_q = np.array(rows)
    scores = ltr_model.predict(X_q)
    idx = np.argsort(scores)[::-1][:top_k]

    return [df_sub.iloc[i] for i in idx], scores[idx].tolist()

def rag_search(query: str, top_k=3):
    print(f"🎵 Query: \"{query}\"\n")

    # 1. Extract metric filters
    filters = extract_filters(query)
    if filters:
        print(f"📊 Filters: {filters}")
        valid_ids = apply_metric_filters(filters)
        print(f"   {len(valid_ids)} documents passed the filter\n")
    else:
        valid_ids = df['doc_id'].tolist()

    # 2. LTR ranking on filtered docs
    top_rows, scores = rank_ltr_filtered(query, valid_ids, top_k=top_k)

    # 3. Build context
    retrieved_docs = []
    context = ""
    for i, (row, score) in enumerate(zip(top_rows, scores), 1):
        doc = documents[int(row['doc_id'])]
        title   = doc.get('title', '')
        artist  = doc['artist'].get('name', '').title()
        genre   = doc.get('genre', '')
        summary = doc['reviews'][0]['summary'] if doc.get('reviews') else ''

        retrieved_docs.append({'title': title, 'artist': artist, 'genre': genre, 'score': round(float(score), 4),'music_id': int(row['doc_id'])})
        context += f"{i}. \"{title}\" by {artist} ({genre})\n"
        if summary:
            context += f"   Review: {summary}\n"
        context += "\n"

    # 4. Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a music search assistant. Answer the user's query using only the retrieved songs provided. Be concise and helpful."},
            {"role": "user", "content": f"Query: {query}\n\nRetrieved songs:\n{context}\nAnswer the query based on the songs above."}
        ],
        max_tokens=300
    )

    # 5. Show results
# 5. Show results
    print("=== Retrieved Documents ===")
    for d in retrieved_docs:
        print(f"  - \"{d['title']}\" by {d['artist']} (score: {d['score']})")
        if filters:
            m = df_measure_docs[df_measure_docs['music_id'] == d['music_id']]
            if not m.empty:
                for field in filters.keys():
                    if field in m.columns:
                        val = round(float(m[field].values[0] or 0), 3)
                        print(f"      {field}: {val}")
    print(f"\n=== Generated Answer ===")
    print(response.choices[0].message.content)
    print("\n")

# Testar
rag_search("sad rock song about love")
rag_search("energetic hip hop with clever lyrics")
rag_search("calm acoustic songs with good reviews")

🎵 Query: "sad rock song about love"

📊 Filters: {'energy': {'lt': 0.4}, 'speechiness': {'lt': 0.5}}
   41166 documents passed the filter

=== Retrieved Documents ===
  - "A Pearl" by Mitski (score: 0.4005)
      energy: 0.364
      speechiness: 0.029
  - "goodnight n go" by Ariana Grande (score: 0.3958)
      energy: 0.279
      speechiness: 0.037
  - "ghostin" by Ariana Grande (score: 0.0697)
      energy: 0.358
      speechiness: 0.029

=== Generated Answer ===
The retrieved song that matches your query for a sad rock song about love is "A Pearl" by Mitski. It's a rock song, and although the review doesn't explicitly mention it being sad, Mitski is known for her emotional and often melancholic songwriting, which could fit the theme of a sad love song.


🎵 Query: "energetic hip hop with clever lyrics"

📊 Filters: {'energy': {'gt': 0.7}, 'speechiness': {'gt': 0.5}}
   672 documents passed the filter

=== Retrieved Documents ===
  - "All Falls Down" by Kanye West (score: 0.0007)
      e